# Deduplication

## Overview
Duplicate rows are one of the most common data quality issues in healthcare and administrative data — caused by double-submissions, ETL reruns, or merged source systems.

**Two distinct problems:**

| Problem | Approach |
|---|---|
| Fully identical rows (every column the same) | `DISTINCT` or `GROUP BY` |
| Near-duplicate rows (same key, minor differences) | `ROW_NUMBER()` with a tie-breaking rule |

**Deduplication strategy:**
1. **Identify** duplicates first — never delete without reviewing
2. **Define the key** — which columns constitute "the same record"?
3. **Define the keeper** — which duplicate should survive (most recent, most complete, lowest ID)?
4. **Isolate** — SELECT duplicates into a review table
5. **Remove** — DELETE using a CTE or subquery that keeps only the chosen row

**PostgreSQL note:** `ON CONFLICT DO NOTHING` / `ON CONFLICT DO UPDATE` (UPSERT) prevents duplicates at insert time. `DISTINCT ON (col)` is a PostgreSQL extension for selecting the first row per group.

---

In [1]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE intake_raw (record_id INTEGER PRIMARY KEY, patient_ref TEXT, first_name TEXT, last_name TEXT, dob TEXT, sex TEXT, province TEXT, phone TEXT, email TEXT, cost_str TEXT, intake_date TEXT, notes TEXT);
CREATE TABLE lab_raw (lab_id INTEGER PRIMARY KEY, patient_ref TEXT, test_code TEXT, result_txt TEXT, collected TEXT, status TEXT);
CREATE TABLE provider_raw (prov_id INTEGER PRIMARY KEY, name_raw TEXT, dept_code TEXT, start_dt TEXT, salary TEXT);
INSERT INTO intake_raw VALUES
  (1,'P-001','aroha','Ngata','1985-03-12','F','NB','506-555-0101','aroha@mail.com','$3,200.00','2023-01-15','Referred by GP'),
  (2,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (3,'P-003','FATIMA','AL-RASHID','1990-07-22','female','Ontario','416-555-0303',NULL,'120.00','2023-03-05','Annual checkup'),
  (4,'P-004','James','MacLeod','Jan 30 1955','M','NB',NULL,'james@mail.com','$5,500','18-03-2023','Surgery follow-up'),
  (5,'P-005','sofia','Petrov','2001-09-15','F','BC','604-555-0505','sofia@mail.com','$95.00','2023-04-02',NULL),
  (6,'P-006','Noah','Williams','08/06/1968','m','AB ','780-555-0606','noah@mail.com','780','11/05/2023',NULL),
  (7,'P-007','Mei','Zhang','1995-04-17','F','ON','416-555-0707',NULL,'$2,100.00','22-06-2023','Follow-up required'),
  (8,'P-002','  Liam ','Chen','04/11/1972','Male','NS','902-555-0202','liam@mail.com','1850','15/02/2023',NULL),
  (9,'P-008','ethan','tremblay','01/12/1980',NULL,'QC','514-555-0808','ethan@mail.com','80.00','14-07-2023',NULL),
  (10,'P-009','Priya','Nair','1977-08-25','F','bc',NULL,'priya@mail.com','$1,750.00','01/10/2023',NULL),
  (11,'P-010','Marcus','Okafor','1993-05-19','M','ON','647-555-1010',NULL,'2200','03-11-2023',NULL),
  (12,'P-011','Diana','Flores','14/02/2000','female','NB','506-555-1111','diana@mail.com',NULL,'2023-12-01',NULL),
  (13,NULL,'Unknown',NULL,NULL,NULL,NULL,NULL,NULL,NULL,NULL,'Incomplete record'),
  (14,'','Test','Record','2023-01-01','X','XX','000-000-0000','test@test.com','-1','2023-01-01','Test entry');
INSERT INTO lab_raw VALUES
  (1,'P-001','HBA1C','7.2 %','2023-03-10','Active'),
  (2,'P-002','HBA1C','9.1%','2023-03-15','active'),
  (3,'P-003','CREAT','88.5umol/L','2023-04-01','ACTIVE'),
  (4,'P-004','CREAT','145 umol/L','2023-04-12','Active'),
  (5,'P-005','HBA1C','5.8 %','2023-05-05',''),
  (6,'P-006','SODIUM','138mmol/L','2023-05-20','Active'),
  (7,'P-007','SODIUM','151 mmol/L','2023-06-01',NULL),
  (8,'P-001','CREAT','72.3 umol/L','2023-06-18','Active'),
  (9,'P-008','HBA1C','6.4%','2023-07-02','Active'),
  (10,'P-009','CREAT','210umol/L','2023-07-15','active');
INSERT INTO provider_raw VALUES
  (1,'MacNeil, Sarah MD','CARD','2018-01-15','$95,000'),
  (2,'Dr. James Wong','ONCO','2019-03-01','88000'),
  (3,'Fatima Osei M.D.','FAM','2017-06-01','$82,500.00'),
  (4,'Larson, Ethan','ORTH','2020-09-15','91000.00'),
  (5,'Sharma, Priya MD','EMRG','2016-11-01','$78,000'),
  (6,'Lucas Petit','RAD','2021-02-28',NULL);
""")
conn.commit()
print("intake_raw rows:", conn.execute("SELECT COUNT(*) FROM intake_raw").fetchone()[0])

intake_raw rows: 14


---
## Identifying duplicates — detect before deleting

In [2]:
# Count how many patient_refs appear more than once
print("=== patient_ref values appearing more than once ===")
q = """
SELECT  patient_ref,
        COUNT(*)  AS occurrences
FROM    intake_raw
WHERE   patient_ref IS NOT NULL AND patient_ref != ''
GROUP BY patient_ref
HAVING  COUNT(*) > 1
ORDER BY occurrences DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Show the actual duplicate rows side by side
print()
print("=== All rows for duplicated patient_refs ===")
q2 = """
SELECT  i.record_id, i.patient_ref, i.first_name, i.last_name,
        i.dob, i.intake_date
FROM    intake_raw AS i
WHERE   i.patient_ref IN (
    SELECT patient_ref FROM intake_raw
    WHERE  patient_ref IS NOT NULL AND patient_ref != ''
    GROUP BY patient_ref HAVING COUNT(*) > 1
)
ORDER BY i.patient_ref, i.record_id
"""
print(pd.read_sql(q2, conn).to_string(index=False))

=== patient_ref values appearing more than once ===
patient_ref  occurrences
      P-002            2

=== All rows for duplicated patient_refs ===
 record_id patient_ref first_name last_name        dob intake_date
         2       P-002      Liam       Chen 04/11/1972  15/02/2023
         8       P-002      Liam       Chen 04/11/1972  15/02/2023


---
## ROW_NUMBER deduplication — keep the first record

In [3]:
# Assign row number within each patient_ref group, keep rn=1
print("=== Dedup with ROW_NUMBER: keep earliest record per patient_ref ===")
q = """
SELECT record_id, patient_ref, first_name, last_name, intake_date, rn
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY patient_ref
               ORDER BY record_id ASC   -- lowest record_id = earliest
           ) AS rn
    FROM intake_raw
    WHERE patient_ref IS NOT NULL AND patient_ref != ''
)
WHERE rn = 1
ORDER BY patient_ref
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Alternative: keep the most recent intake_date
print()
print("=== Dedup: keep most recent intake_date per patient_ref ===")
q2 = """
SELECT record_id, patient_ref, first_name, last_name, intake_date, rn
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY patient_ref
               ORDER BY intake_date DESC, record_id DESC
           ) AS rn
    FROM intake_raw
    WHERE patient_ref IS NOT NULL AND patient_ref != ''
)
WHERE rn = 1
ORDER BY patient_ref
"""
print(pd.read_sql(q2, conn).to_string(index=False))

=== Dedup with ROW_NUMBER: keep earliest record per patient_ref ===
 record_id patient_ref first_name last_name intake_date  rn
         1       P-001      aroha     Ngata  2023-01-15   1
         2       P-002      Liam       Chen  15/02/2023   1
         3       P-003     FATIMA AL-RASHID  2023-03-05   1
         4       P-004      James   MacLeod  18-03-2023   1
         5       P-005      sofia    Petrov  2023-04-02   1
         6       P-006       Noah  Williams  11/05/2023   1
         7       P-007        Mei     Zhang  22-06-2023   1
         9       P-008      ethan  tremblay  14-07-2023   1
        10       P-009      Priya      Nair  01/10/2023   1
        11       P-010     Marcus    Okafor  03-11-2023   1
        12       P-011      Diana    Flores  2023-12-01   1

=== Dedup: keep most recent intake_date per patient_ref ===
 record_id patient_ref first_name last_name intake_date  rn
         1       P-001      aroha     Ngata  2023-01-15   1
         8       P-002      Lia

---
## DISTINCT and GROUP BY for fully identical rows

In [4]:
# DISTINCT removes rows where every selected column is identical
print("=== DISTINCT on key columns ===")
q = """
SELECT DISTINCT patient_ref, first_name, last_name, dob, sex
FROM   intake_raw
WHERE  patient_ref IS NOT NULL AND patient_ref != ''
ORDER BY patient_ref
"""
result = pd.read_sql(q, conn)
print(f"Rows after DISTINCT: {len(result)} (was 12 with valid refs)")
print(result.to_string(index=False))

# COUNT to quantify the dedup impact
print()
print("=== Dedup impact summary ===")
q2 = """
SELECT
    COUNT(*)                                   AS total_rows,
    COUNT(DISTINCT patient_ref)                AS distinct_patients,
    COUNT(*) - COUNT(DISTINCT patient_ref)     AS duplicate_rows,
    ROUND(100.0*(COUNT(*)-COUNT(DISTINCT patient_ref))/COUNT(*),1) AS dup_pct
FROM intake_raw
WHERE patient_ref IS NOT NULL AND patient_ref != ''
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()

=== DISTINCT on key columns ===
Rows after DISTINCT: 11 (was 12 with valid refs)
patient_ref first_name last_name         dob    sex
      P-001      aroha     Ngata  1985-03-12      F
      P-002      Liam       Chen  04/11/1972   Male
      P-003     FATIMA AL-RASHID  1990-07-22 female
      P-004      James   MacLeod Jan 30 1955      M
      P-005      sofia    Petrov  2001-09-15      F
      P-006       Noah  Williams  08/06/1968      m
      P-007        Mei     Zhang  1995-04-17      F
      P-008      ethan  tremblay  01/12/1980   None
      P-009      Priya      Nair  1977-08-25      F
      P-010     Marcus    Okafor  1993-05-19      M
      P-011      Diana    Flores  14/02/2000 female

=== Dedup impact summary ===
 total_rows  distinct_patients  duplicate_rows  dup_pct
         12                 11               1      8.3


---
## Common Pitfalls

**1. Deleting duplicates without first reviewing them**
Always SELECT the rows to be deleted before running DELETE. Duplicates in production data sometimes reflect real events (e.g. two separate encounters for the same patient on the same day). Review before removing.

**2. Defining the wrong uniqueness key**
"Same patient" might mean same `patient_ref`, or same `(first_name, last_name, dob)`, or same `email`. Choosing the wrong key either over-deduplicates (merging genuinely different people) or under-deduplicates (missing real duplicates). Define the key based on domain knowledge.

**3. Using DISTINCT when you need ROW_NUMBER**
`SELECT DISTINCT patient_ref, first_name` returns one row per distinct (patient_ref, first_name) combination — not one row per patient. If the same patient has two different first_name spellings, both rows survive. `ROW_NUMBER() OVER (PARTITION BY patient_ref)` gives you explicit control over which row is kept.

**4. ROW_NUMBER ties produce non-deterministic results without a tiebreaker**
`ROW_NUMBER() OVER (PARTITION BY patient_ref ORDER BY intake_date)` is non-deterministic when two records share the same date. Always add a unique column (`record_id`) as a tiebreaker.

**5. Deduplication that loses data from the discarded rows**
When keeping the most recent record, earlier records may contain useful information (earlier diagnoses, different contact details). Consider merging rather than discarding: keep the most recent record but backfill missing fields from earlier duplicates using COALESCE.

**6. Not accounting for near-duplicates with typos**
`'P-002'` and `'P-0O2'` (letter O vs zero) look like different patient refs. Exact deduplication misses these. Consider normalising the key before deduplication, or using fuzzy matching at the application layer.


---
*sql_methods_library - Samantha McGarrigle*